# MetLife Stadium - Ookla Q3 2025 Mobile Performance Analysis

This notebook downloads Ookla Q3 2025 mobile performance data, filters for samples near MetLife Stadium, and creates an interactive Mapbox visualization.

**MetLife Stadium Coordinates:** 40.813778°N, 74.074310°W

## How to Run
1. Click **Runtime > Run all** or run cells sequentially
2. The visualization will be displayed inline and saved as HTML
3. Download the HTML file for offline viewing

## 1. Install Dependencies

In [ ]:
!pip install -q duckdb pandas plotly shapely pyproj

## 2. Configuration

In [ ]:
import os
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import HTML, display
import math

# Mapbox Token
MAPBOX_TOKEN = "pk.eyJ1IjoiYXpoYXJ6NHUiLCJhIjoiY2pkaHFtbHAxMGV1cDJxbzI0cjFlcWt4eiJ9.D-0A_N0JhPBfOm-CeFZtMQ"

# MetLife Stadium coordinates
METLIFE_LAT = 40.813778
METLIFE_LON = -74.074310

# Search radius in kilometers
SEARCH_RADIUS_KM = 0.5

# Ookla data paths
OOKLA_S3_PATH = "s3://ookla-open-data/parquet/performance/type=mobile/year=2025/quarter=3/2025-07-01_performance_mobile_tiles.parquet"
LOCAL_PARQUET_PATH = "2025-07-01_performance_mobile_tiles.parquet"

# Stadium boundary polygon
METLIFE_STADIUM_POLYGON = [
    [-74.0780, 40.8118],
    [-74.0780, 40.8158],
    [-74.0705, 40.8158],
    [-74.0705, 40.8118],
    [-74.0780, 40.8118],
]

print(f"MetLife Stadium: {METLIFE_LAT}°N, {abs(METLIFE_LON)}°W")
print(f"Search radius: {SEARCH_RADIUS_KM * 1000}m")

## 3. Download Ookla Q3 2025 Data

In [ ]:
%%time

# Download the Ookla parquet file (~185MB)
if not os.path.exists(LOCAL_PARQUET_PATH):
    print("Downloading Ookla Q3 2025 mobile performance data...")
    print("File size: ~185 MB - this may take a few minutes")
    !aws s3 cp {OOKLA_S3_PATH} {LOCAL_PARQUET_PATH} --no-sign-request
else:
    file_size = os.path.getsize(LOCAL_PARQUET_PATH) / (1024*1024)
    print(f"File already exists: {LOCAL_PARQUET_PATH} ({file_size:.1f} MB)")

## 4. Helper Functions

In [ ]:
def quadkey_to_tile(quadkey):
    """Convert quadkey to tile coordinates (x, y, zoom)."""
    x = y = 0
    zoom = len(quadkey)
    for i, char in enumerate(quadkey):
        bit = zoom - i - 1
        mask = 1 << bit
        if char == '1':
            x |= mask
        elif char == '2':
            y |= mask
        elif char == '3':
            x |= mask
            y |= mask
    return x, y, zoom


def tile_to_bbox(x, y, zoom):
    """Convert tile coordinates to bounding box."""
    n = 2 ** zoom
    lon_min = x / n * 360.0 - 180.0
    lon_max = (x + 1) / n * 360.0 - 180.0
    lat_max = math.degrees(math.atan(math.sinh(math.pi * (1 - 2 * y / n))))
    lat_min = math.degrees(math.atan(math.sinh(math.pi * (1 - 2 * (y + 1) / n))))
    return lon_min, lat_min, lon_max, lat_max


def quadkey_to_center(quadkey):
    """Convert quadkey to center point (lat, lon)."""
    x, y, zoom = quadkey_to_tile(quadkey)
    lon_min, lat_min, lon_max, lat_max = tile_to_bbox(x, y, zoom)
    return (lat_min + lat_max) / 2, (lon_min + lon_max) / 2


def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate distance between two points in km."""
    R = 6371
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))


print("Helper functions loaded successfully!")

## 5. Load and Filter Data with DuckDB

In [ ]:
%%time

# Connect to DuckDB and load the parquet file
con = duckdb.connect()

# Check the schema
print("Parquet Schema:")
schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{LOCAL_PARQUET_PATH}')").fetchdf()
display(schema)

# Get total count
total_count = con.execute(f"SELECT COUNT(*) FROM read_parquet('{LOCAL_PARQUET_PATH}')").fetchone()[0]
print(f"\nTotal records: {total_count:,}")

# Sample data
print("\nSample data:")
sample = con.execute(f"SELECT * FROM read_parquet('{LOCAL_PARQUET_PATH}') LIMIT 5").fetchdf()
display(sample)

In [ ]:
%%time

# Load all quadkeys and their data
print("Loading data from parquet...")
all_data = con.execute(f"""
    SELECT
        quadkey,
        avg_d_kbps,
        avg_u_kbps,
        avg_lat_ms,
        tests,
        devices
    FROM read_parquet('{LOCAL_PARQUET_PATH}')
    WHERE quadkey IS NOT NULL
""").fetchdf()

print(f"Loaded {len(all_data):,} records with quadkeys")
con.close()

In [ ]:
%%time

# Filter for MetLife Stadium area
print(f"\nFiltering for points within {SEARCH_RADIUS_KM * 1000}m of MetLife Stadium...")

filtered_records = []
for idx, row in all_data.iterrows():
    if idx % 500000 == 0:
        print(f"  Processing record {idx:,}/{len(all_data):,}...")
    try:
        quadkey = str(row['quadkey'])
        center_lat, center_lon = quadkey_to_center(quadkey)
        distance = haversine_distance(METLIFE_LAT, METLIFE_LON, center_lat, center_lon)
        
        if distance <= SEARCH_RADIUS_KM:
            filtered_records.append({
                'quadkey': quadkey,
                'lat': center_lat,
                'lon': center_lon,
                'avg_d_kbps': row['avg_d_kbps'],
                'avg_u_kbps': row['avg_u_kbps'],
                'avg_lat_ms': row['avg_lat_ms'],
                'tests': row['tests'],
                'devices': row['devices'],
                'distance_km': distance,
                'download_mbps': row['avg_d_kbps'] / 1000 if pd.notna(row['avg_d_kbps']) else None,
                'upload_mbps': row['avg_u_kbps'] / 1000 if pd.notna(row['avg_u_kbps']) else None
            })
    except:
        continue

df = pd.DataFrame(filtered_records)
print(f"\nFound {len(df)} tiles within {SEARCH_RADIUS_KM * 1000}m of MetLife Stadium!")

if len(df) > 0:
    display(df)

## 6. Create Mapbox Visualization

In [ ]:
# Create the visualization
fig = go.Figure()

# Add stadium polygon outline
stadium_lons = [p[0] for p in METLIFE_STADIUM_POLYGON]
stadium_lats = [p[1] for p in METLIFE_STADIUM_POLYGON]

fig.add_trace(go.Scattermapbox(
    name="MetLife Stadium",
    mode="lines",
    lon=stadium_lons,
    lat=stadium_lats,
    line=dict(width=3, color='blue'),
    fill='toself',
    fillcolor='rgba(0, 0, 255, 0.1)',
    hoverinfo='name'
))

# Add stadium center marker
fig.add_trace(go.Scattermapbox(
    name="Stadium Center",
    mode="markers+text",
    lon=[METLIFE_LON],
    lat=[METLIFE_LAT],
    marker=dict(size=15, color='red'),
    text=["MetLife Stadium"],
    textposition="top center",
    hovertemplate="<b>MetLife Stadium</b><br>Lat: %{lat:.6f}<br>Lon: %{lon:.6f}<extra></extra>"
))

# Add Ookla data points
if len(df) > 0:
    fig.add_trace(go.Scattermapbox(
        name="Ookla Speed Tests",
        mode="markers",
        lon=df['lon'],
        lat=df['lat'],
        marker=dict(
            size=14,
            color=df['download_mbps'],
            colorscale='RdYlGn',
            cmin=0,
            cmax=df['download_mbps'].quantile(0.95),
            colorbar=dict(title="Download<br>Speed (Mbps)", x=1.02),
            opacity=0.8
        ),
        text=df.apply(lambda r: 
            f"Download: {r['download_mbps']:.1f} Mbps<br>"
            f"Upload: {r['upload_mbps']:.1f} Mbps<br>"
            f"Latency: {r['avg_lat_ms']:.0f} ms<br>"
            f"Tests: {r['tests']}<br>"
            f"Distance: {r['distance_km']*1000:.0f}m", axis=1),
        hovertemplate="<b>Ookla Tile</b><br>%{text}<extra></extra>"
    ))

# Configure layout
fig.update_layout(
    title=dict(
        text="<b>MetLife Stadium - Ookla Q3 2025 Mobile Performance</b>",
        x=0.5,
        font=dict(size=18)
    ),
    mapbox=dict(
        accesstoken=MAPBOX_TOKEN,
        style="mapbox://styles/mapbox/satellite-streets-v12",
        center=dict(lat=METLIFE_LAT, lon=METLIFE_LON),
        zoom=15,
    ),
    showlegend=True,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01, bgcolor="rgba(255,255,255,0.8)"),
    margin=dict(l=0, r=0, t=50, b=0),
    height=700
)

# Display the figure
fig.show()

## 7. Statistics Summary

In [ ]:
if len(df) > 0:
    print("=" * 50)
    print("METLIFE STADIUM - Q3 2025 NETWORK PERFORMANCE")
    print("=" * 50)
    print(f"\nCoverage Tiles: {len(df)}")
    print(f"Total Speed Tests: {df['tests'].sum():,}")
    print(f"Total Devices: {df['devices'].sum():,}")
    print(f"\n--- Download Speeds ---")
    print(f"  Average: {df['download_mbps'].mean():.1f} Mbps")
    print(f"  Maximum: {df['download_mbps'].max():.1f} Mbps")
    print(f"  Minimum: {df['download_mbps'].min():.1f} Mbps")
    print(f"\n--- Upload Speeds ---")
    print(f"  Average: {df['upload_mbps'].mean():.1f} Mbps")
    print(f"  Maximum: {df['upload_mbps'].max():.1f} Mbps")
    print(f"\n--- Latency ---")
    print(f"  Average: {df['avg_lat_ms'].mean():.0f} ms")
    print(f"  Minimum: {df['avg_lat_ms'].min():.0f} ms")
    print("=" * 50)
else:
    print("No data found within the search radius.")

## 8. Save HTML File

In [ ]:
# Save the visualization as HTML
output_file = "metlife_stadium_ookla_q3_2025.html"
fig.write_html(output_file)
print(f"Saved visualization to: {output_file}")

# Download link for Colab
try:
    from google.colab import files
    files.download(output_file)
    print("Download started!")
except:
    print(f"Open {output_file} in your browser to view the visualization.")

## 9. Save Data to CSV

In [ ]:
if len(df) > 0:
    csv_file = "metlife_stadium_ookla_data.csv"
    df.to_csv(csv_file, index=False)
    print(f"Saved data to: {csv_file}")
    
    # Download link for Colab
    try:
        from google.colab import files
        files.download(csv_file)
    except:
        pass